In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("../utils"))

from pathlib import Path
import re
import torch as th
from imitation.data import rollout
from imitation.data.types import Trajectory
from imitation.data import rollout
import numpy as np
from imitation.algorithms import bc
import gymnasium as gym
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.policies import ActorCriticPolicy
import torch.nn as nn
from imitation.util import logger as imit_logger
from imitation.algorithms.bc import BC
from torch.utils.data import DataLoader
from imitation.data.types import Transitions
import datetime

from resident_requiem import TemporalAttentionLSTM
import gc
from IPython import get_ipython
import cv2

from utils import get_last_index

current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_path = f"./tensorboard_logs/run_{current_time}"
custom_logger = imit_logger.configure(log_path, ["tensorboard", "stdout"])

gc.collect()
th.cuda.empty_cache()

rng = np.random.default_rng()

demo_path = 'demos/'

train_path = 'imitation/'
sub_train_path = train_path + "steps/"

os.makedirs(train_path, exist_ok=True)
os.makedirs(sub_train_path, exist_ok=True)


ENT_WEIGHT= 1e-3 # 0 # 1e-4 # 0.001 # 0.01 # 0 # 1e-3 # 0 # 1e-4
BATCH_SIZE= 384 # 512 # 128 # 128 # 256 # 64 # 256 # 128 # 64 # 128 # 32 # 64 # 128
MINI_BATCH= None # 128 # 64 # 32 # None # 128 # None # 64
NUMBER_OF_EPOCH= 40 # 30 # 60 # 75 # 15 #45 # 20 # 45 # 35 # 30 # 27
EPOCH_PER_FILE= 2 # 2 # 1 #3 # 1 # 2 # 1 # 3 # 2
L2= 1e-5 # 1e-4 # 1e-3 # 1e-4 # 0 # 1e-4 # 1e-5
LEARNING_RATE= 1e-4 # 4e-4  # 1e-4 # 5e-5 # 1e-4 # 2e-4
BUFFER_SIZE = 4 # 4 # 3 # 10 # 5 # 5 # 10 # 7 # 4 # 7
BUFFER_SIZE_ORIG = BUFFER_SIZE
IS_HG_DAGGER=True

if IS_HG_DAGGER:
    # TODO: set to 10 when training dagger
    NUMBER_OF_EPOCH = 5 # 10 # 5 # 20 # 10 # 30 # 10

action_space = gym.spaces.MultiBinary(18)

observation_space = gym.spaces.Box(
    low=0,
    high=255,
    shape=(1, 128, 128), # 1 frames 128x128
    dtype=np.uint8,
)


last_index = get_last_index(demo_path, "demos", "pt")

print("last_index: " + str(last_index))

files_index = np.arange(last_index + 1)

class BCNoShuffle(BC):
    def _make_data_loader(self, transitions, batch_size):
        dataset = self._make_dataset(transitions)
        
        return DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=False,
            drop_last=True,
        )

def clean_combat_logic(acts):
    # Ensures that in the training dataset, shooting only exists with aiming
    # acts[:, 7] is Aim, acts[:, 8] is Shoot
    invalid_shots = (acts[:, 8] == 1) & (acts[:, 7] == 0)
    acts[invalid_shots, 8] = 0
    return acts

def manual_flatten_trajectories(trajectories):
    all_obs = []
    all_acts = []
    all_next_obs = []
    all_dones = []
    all_infos = []

    for traj in trajectories:
        all_obs.append(traj.obs[:-1])
        
        all_acts.append(traj.acts)
        
        all_next_obs.append(traj.obs[1:])
        
        dones = np.zeros(len(traj.acts), dtype=bool)
        dones[-1] = True
        all_dones.append(dones)
        
        if traj.infos is not None:
            all_infos.extend(traj.infos)
        else:
            all_infos.extend([{}] * len(traj.acts))

    return Transitions(
        obs=np.concatenate(all_obs, axis=0),
        acts=np.concatenate(all_acts, axis=0),
        next_obs=np.concatenate(all_next_obs, axis=0),
        dones=np.concatenate(all_dones, axis=0),
        infos=np.array(all_infos)
    )

def split_into_traj(trajectories, num_parts=4):
    split_trajs = []
    for traj in trajectories:
        total_frames = len(traj.acts)
        part_duration = total_frames // num_parts
        
        for i in range(num_parts):
            start = i * part_duration
            end = (i + 1) * part_duration if i < (num_parts - 1) else total_frames
            
            chunk_obs = traj.obs[start : end + 1]
            chunk_acts = traj.acts[start : end]
            
            split_trajs.append(
                Trajectory(
                    obs=chunk_obs,
                    acts=chunk_acts,
                    infos=traj.infos[start : end] if traj.infos else None,
                    terminal=(i == num_parts - 1)
                )
            )
    return split_trajs

class CustomActorCriticPolicy(ActorCriticPolicy):
    def __init__(self, *args, **kwargs):
        super().__init__(
            *args,
            **kwargs,
            features_extractor_class=TemporalAttentionLSTM,
            features_extractor_kwargs=dict(features_dim=256),
            net_arch=dict(pi=[256, 256], vf=[256, 256])
        )
        self._last_dones = None
        self.retain_graph = True
    
    def forward(self, obs, deterministic=False):
        if hasattr(self.features_extractor, 'reset_hidden'):
            if self._last_dones is not None:
                self.features_extractor.reset_hidden(self._last_dones)
        
        return super().forward(obs, deterministic)
    
    def predict(self, observation, state=None, episode_start=None, deterministic=False):
        if episode_start is not None:
            self._last_dones = episode_start
        
        return super().predict(observation, state, episode_start, deterministic)


policy = CustomActorCriticPolicy(
    observation_space=observation_space,
    action_space=action_space,
    lr_schedule=lambda _: th.finfo(th.float32).max,  # BC control the learning rate
)


th.serialization.add_safe_globals([Trajectory])

last_index_imitation = get_last_index(str(sub_train_path), "bc_policy", "zip")

if IS_HG_DAGGER:
    policy = ActorCriticPolicy.load(f"./imitation/steps/bc_policy{last_index_imitation}.zip")


bc_trainer = BCNoShuffle(
    observation_space=observation_space,
    action_space=action_space,
    rng=rng,
    ent_weight=ENT_WEIGHT,
    batch_size=BATCH_SIZE,
    policy=policy,
    optimizer_kwargs=dict(lr=LEARNING_RATE),
    l2_weight=L2,
    minibatch_size=MINI_BATCH,
    custom_logger=custom_logger,
)


def fix_action_format(acts):
    """
    Fix action shape
    """
    if isinstance(acts, np.ndarray):
        if acts.ndim == 3 and acts.shape[1] == 1:
            acts = acts.squeeze(1)
        
        acts = acts.astype(np.float32)
    
    return acts


# TODO set zero
epoch_count = 0 # 72 # 0

def augment_brightness(obs):
    factor = np.random.uniform(0.8, 1.2)

    obs = np.clip(obs.astype(np.uint16) * factor, 0, 255).astype(np.uint8)

    return obs

def augment_noise(obs):

    noise = np.random.normal(0, 5, obs.shape)

    obs = obs.astype(np.float32) + noise
    obs = np.clip(obs, 0, 255)

    return obs.astype(np.uint8)

def random_crop(obs, crop=4):

    T, C, H, W = obs.shape

    y = np.random.randint(0, crop)
    x = np.random.randint(0, crop)

    cropped = obs[:, :, y:H-(crop-y), x:W-(crop-x)]

    resized = np.zeros((T, C, H, W), dtype=obs.dtype)

    for t in range(T):
        for c in range(C):
            resized[t,c] = cv2.resize(cropped[t,c], (W,H))

    return resized

def smooth_action(acts, act_idx, window=10):
    smoothed_acts = acts.copy()
    for t in range(len(smoothed_acts) - window):
        if smoothed_acts[t, act_idx] == 1:
            smoothed_acts[t:t+window, act_idx] = 1
    return smoothed_acts


def fix_trajectories(trajectories, aug_prob=0.5):
    fixed_trajectories = []

    for traj in trajectories:
        obs = np.array(traj.obs)

        acts = fix_action_format(np.array(traj.acts, dtype=np.int8))

        # clean fire button without aim
        acts = clean_combat_logic(acts)

        # fix run press
        acts = smooth_action(acts, act_idx=9)

        # fix fire press
        # window = np.random.randint(0, 5)
        # acts = smooth_action(acts, act_idx=8, window=window)

        # Case (T, 1, H, W, C)
        if obs.ndim == 5 and obs.shape[1] == 1:
            obs = obs[:, 0]  # remove dimension 1 → (T, H, W, C)
        
        # Case HWC → CHW
        if obs.shape[-1] == 1: # obs.shape[-1] == 4
            obs = obs.transpose(0, 3, 1, 2)  # (T, 1, H, W)

        # print("obs shape", obs.shape)

        if obs.shape[0]>= BATCH_SIZE * 10:
            print("obs shape:", obs.shape)

            fixed_trajectories.append(
                Trajectory(
                    obs=obs,
                    acts=acts,
                    infos=traj.infos,
                    terminal=traj.terminal
                )
            )

            aug_obs = obs
            
            if np.random.random() < aug_prob:
                aug_type = np.random.choice(['brightness', 'crop', 'noise'])

                # augment fire press
                window = np.random.randint(0, 5)
                acts = smooth_action(acts, act_idx=8, window=window)

                if aug_type == 'brightness':
                    aug_obs = augment_brightness(obs)
                elif aug_type == 'crop':
                    aug_obs = random_crop(obs)
                elif aug_type == 'noise':
                    aug_obs = augment_noise(obs)

                fixed_trajectories.append(Trajectory(obs=aug_obs, acts=acts, infos=traj.infos, terminal=traj.terminal))

    return fixed_trajectories


for e in range(NUMBER_OF_EPOCH):
    np.random.shuffle(files_index)
    
    BUFFER_SIZE = BUFFER_SIZE_ORIG

    if IS_HG_DAGGER:
        epoch_count = last_index_imitation + 1
    else:
        epoch_count += 1

    if IS_HG_DAGGER:
        print(f"\n--------------- Dagger pass: {e} ------------------\n")
    else:
        print(f"\n--------------- Epoch: {epoch_count} ------------------\n")
    print("files_index: ", files_index)
    
    buffer = []
    buffer_files = []
    
    for idx, file_idx in enumerate(files_index):
 
        trajectories = th.load(demo_path + f"demos{file_idx}.pt", weights_only=False)
        fixed_trajectories = fix_trajectories(trajectories)
        np.random.shuffle(fixed_trajectories)

        buffer.append(fixed_trajectories)
        buffer_files.append(file_idx)
        
        # del transitions, fixed_trajectories, trajectories
        del fixed_trajectories, trajectories


        if len(buffer) == BUFFER_SIZE or (idx == len(files_index) - 1 and len(buffer) > 0):
            if idx == len(files_index) - 1 and len(buffer) < BUFFER_SIZE:
                print(f"Last batch {len(buffer)} files (less than BUFFER_SIZE)")
                
                diff = [item for item in files_index if item not in buffer_files]
                
                samples = rng.choice(diff, size=(BUFFER_SIZE - len(buffer)), replace=False)

                print("Last batch add new files: ", samples)

                for index in samples:
                    trajectories = th.load(demo_path + f"demos{index}.pt", weights_only=False)
                    fixed_trajectories = fix_trajectories(trajectories)
                    np.random.shuffle(fixed_trajectories)
            
                    buffer.append(fixed_trajectories)
                    buffer_files.append(file_idx)
            
            print(f"Processing files: {buffer_files}")
            
            np.random.shuffle(buffer)

            # flatten
            buffer = [item for sublist in buffer for item in sublist]
            num_parts = np.random.randint(4, 7) 
            # num_parts = np.random.randint(4, 6) 
            buffer_splited = split_into_traj(buffer, num_parts=num_parts)

            np.random.shuffle(buffer_splited)

            # for traj in buffer:
            for traj in buffer_splited:

                if len(traj.acts) < BATCH_SIZE:
                    continue
                
                # clear LSTM buffer and hidden state
                bc_trainer.policy.features_extractor.reset_hidden()
                
                transition = manual_flatten_trajectories([traj])
                bc_trainer.set_demonstrations(transition)
    
                bc_trainer.train(
                    n_epochs=EPOCH_PER_FILE,
                    progress_bar=True,
                    log_interval=1000,
                )
            
            bc_trainer.policy.save(sub_train_path + f"bc_policy{epoch_count}.zip")
            
            buffer.clear()
            buffer_files.clear()
            th.cuda.empty_cache()
            gc.collect()


# bc_trainer.policy.save(train_path + f"bc_policy{last_index_imitation + 1}.zip")

gc.collect()
bc_trainer._demonstrations = None
bc_trainer._demonstrations_tensor = None
del bc_trainer

th.cuda.empty_cache()

print("Force cell kernel reset")
get_ipython().kernel.do_shutdown(restart=True)

last_index: 25


D:\Python311\Lib\site-packages\stable_baselines3\common\policies.py:176: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_variables = th.load(path, map_location=device)



--------------- Dagger pass: 0 ------------------

files_index:  [19 10 14  8  4  3 16 22 12  6 21 17 18 25 15 24  2  7 20  5  1  0 11 13
  9 23]
obs shape: (5220, 1, 128, 128)
obs shape: (5654, 1, 128, 128)
obs shape: (6443, 1, 128, 128)
obs shape: (4859, 1, 128, 128)
obs shape: (5768, 1, 128, 128)
obs shape: (5268, 1, 128, 128)
obs shape: (5584, 1, 128, 128)
obs shape: (5183, 1, 128, 128)
obs shape: (5690, 1, 128, 128)
obs shape: (5332, 1, 128, 128)
obs shape: (4909, 1, 128, 128)
obs shape: (6224, 1, 128, 128)
obs shape: (5650, 1, 128, 128)
obs shape: (5892, 1, 128, 128)
obs shape: (6193, 1, 128, 128)
obs shape: (4136, 1, 128, 128)
obs shape: (4896, 1, 128, 128)
obs shape: (5083, 1, 128, 128)
obs shape: (5774, 1, 128, 128)
obs shape: (5862, 1, 128, 128)
obs shape: (6043, 1, 128, 128)
obs shape: (5364, 1, 128, 128)
obs shape: (5219, 1, 128, 128)
Processing files: [19, 10, 14, 8]


0batch [00:00, ?batch/s]

--------------------------------
| batch_size        | 384      |
| bc/               |          |
|    batch          | 0        |
|    ent_loss       | -0.00144 |
|    entropy        | 1.44     |
|    epoch          | 0        |
|    l2_loss        | 0.266    |
|    l2_norm        | 2.66e+04 |
|    loss           | 1.62     |
|    neglogp        | 1.36     |
|    prob_true_act  | 0.46     |
|    samples_so_far | 384      |
--------------------------------


...

--------------------------------
| batch_size        | 384      |
| bc/               |          |
|    batch          | 0        |
|    ent_loss       | -0.00138 |
|    entropy        | 1.38     |
|    epoch          | 0        |
|    l2_loss        | 0.265    |
|    l2_norm        | 2.65e+04 |
|    loss           | 1.13     |
|    neglogp        | 0.867    |
|    prob_true_act  | 0.506    |
|    samples_so_far | 384      |
--------------------------------


2batch [00:00, 16.95batch/s]
4batch [00:00, 16.85batch/s]
4batch [00:00, 16.65batch/s]
0batch [00:00, ?batch/s]

---------------------------------
| batch_size        | 384       |
| bc/               |           |
|    batch          | 0         |
|    ent_loss       | -0.000888 |
|    entropy        | 0.888     |
|    epoch          | 0         |
|    l2_loss        | 0.265     |
|    l2_norm        | 2.65e+04  |
|    loss           | 0.93      |
|    neglogp        | 0.665     |
|    prob_true_act  | 0.645     |
|    samples_so_far | 384       |
---------------------------------


1batch [00:00,  1.32batch/s]
3batch [00:00,  4.26batch/s]
4batch [00:00,  4.34batch/s]


obs shape: (6861, 1, 128, 128)
obs shape: (5962, 1, 128, 128)
obs shape: (6481, 1, 128, 128)
obs shape: (6184, 1, 128, 128)
obs shape: (6481, 1, 128, 128)
obs shape: (5602, 1, 128, 128)
obs shape: (6028, 1, 128, 128)
obs shape: (6566, 1, 128, 128)
obs shape: (5622, 1, 128, 128)
obs shape: (6080, 1, 128, 128)
obs shape: (5808, 1, 128, 128)
obs shape: (5447, 1, 128, 128)
obs shape: (5405, 1, 128, 128)
